[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# La lógica del contraste de hipótesis.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · Módulo III: *Inferencia estadística*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En este *notebook* se presentará la lógica del contraste de hipótesis: H0 y H1, el valor *p*, los errores tipo I y II y la potencia de una prueba.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* 🟢 **Ejercicio básico:** lo que todos deben dominar.
* 🔴 **Reto (opcional):** para quien termina antes o quiere profundizar. Si no da tiempo en clase, se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.

## 📚 Referencias.

py311 no cubre inferencia, así que la referencia de la API es la documentación oficial: [*SciPy* · `scipy.stats`](https://docs.scipy.org/doc/scipy/reference/stats.html) (`ttest_ind`, `mannwhitneyu`, `permutation_test`, `chi2_contingency`, `f_oneway`, `levene`), [*statsmodels* · pruebas de proporciones](https://www.statsmodels.org/stable/generated/statsmodels.stats.proportion.proportions_ztest.html), [*statsmodels* · `pairwise_tukeyhsd`](https://www.statsmodels.org/stable/generated/statsmodels.stats.multicomp.pairwise_tukeyhsd.html) y [*statsmodels* · `anova_oneway`](https://www.statsmodels.org/stable/generated/statsmodels.stats.oneway.anova_oneway.html).

### Preparación del entorno.

* La siguiente celda importará las bibliotecas y las funciones de *SciPy* y *statsmodels* que se usarán, y definirá `DATA` y la semilla `rng`.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.oneway import anova_oneway
from statsmodels.stats.power import TTestIndPower
from statsmodels.stats.proportion import proportions_ztest

* La siguiente celda leerá `penguins_limpio.csv` (o lo regenerará si no existe), `titanic.csv` y `tips.csv`, y creará `tip_pct`.

In [ ]:
ruta_limpio = os.path.join(DATA, "penguins_limpio.csv")
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
if not os.path.isfile(ruta_limpio):
    (
        pd.read_csv(os.path.join(DATA, "penguins.csv"))
        .assign(sex=lambda d: d["sex"].str.title())
        .dropna(subset=medidas, how="all")
        .to_csv(ruta_limpio, index=False)
    )
pinguinos = pd.read_csv(ruta_limpio)
titanic = pd.read_csv(os.path.join(DATA, "titanic.csv"))
propinas = pd.read_csv(os.path.join(DATA, "tips.csv")).assign(tip_pct=lambda d: 100 * d["tip"] / d["total_bill"])
pinguinos.shape, titanic.shape, propinas.shape

---
## 1. Repaso (10 min).

En `18_bootstrap_e_interpretacion_de_intervalos.ipynb`, un IC del 95% para la diferencia de masa media entre machos y hembras Gentoo fue de unos 700 a 910 g: el 0 quedó muy lejos. Una prueba de hipótesis responde la pregunta complementaria: **si en la población no hubiera diferencia, ¿qué tan raro sería observar una diferencia como esta?** IC y prueba se complementan; el IC además informa la **magnitud**.

---
## 2. La lógica del contraste (20 min).

| Concepto | Significado |
|---|---|
| H0 (hipótesis nula) | "No hay efecto": diferencia 0, independencia, medias iguales |
| H1 (alternativa) | Hay diferencia (bilateral) o va en una dirección (unilateral, decidida **antes** de ver los datos) |
| Valor *p* | Probabilidad, **si H0 fuera cierta**, de obtener un resultado al menos tan extremo como el observado |
| α (nivel de significancia) | Umbral fijado de antemano (habitualmente 0.05) |
| Error tipo I | Rechazar H0 siendo cierta (probabilidad α) |
| Error tipo II | No rechazar H0 siendo falsa (probabilidad β) |
| Potencia | 1 − β: probabilidad de detectar un efecto que sí existe |

**Advertencia:** el valor *p* **no** es la probabilidad de que H0 sea cierta, no mide el tamaño del efecto y *p* > 0.05 **no** demuestra que no haya diferencia.

**Concepto: los dos errores, con una alarma de humo.** Piensa en una alarma de humo; su hipótesis nula (H0) es «no hay incendio».
* El **error tipo I** es que suene sin incendio: una falsa alarma. Su probabilidad es α, que se fija de antemano.
* El **error tipo II** es que **no** suene cuando sí hay incendio. La **potencia** es la probabilidad de que sí suene cuando sí hay incendio (1 − β).

Bajar α para tener menos falsas alarmas hace más probable que la alarma se calle ante un incendio pequeño: los dos errores se compensan, salvo que se tengan más datos (un detector mejor): con más datos, β baja sin que α suba.

| | **H0 es cierta** (no hay efecto) | **H0 es falsa** (sí hay efecto) |
|---|---|---|
| **No se rechaza H0** | Decisión correcta | Error tipo II (probabilidad β) |
| **Se rechaza H0** | Error tipo I (probabilidad α) | Decisión correcta (potencia = 1 − β) |

**¿Por qué 0.05?** Es una convención, no una ley: significa aceptar que, de cada 20 veces que no hay nada, se declare un hallazgo una. En contextos de más riesgo se usa un umbral menor.

**Ejemplo de «*p* > 0.05 no demuestra que no haya diferencia»:** los Chinstrap y los Adelie difieren en masa media por solo 32 g (3,733 contra 3,701 g) y la prueba da *p* = 0.59. Pero el intervalo de confianza de la diferencia (calculado con el método de Welch) va de −86 a +150 g: los datos no descartan diferencias de hasta unos 150 g.

**Concepto: cómo se construye una prueba.** Toda prueba de hipótesis sigue los mismos cuatro pasos:

1. Plantear H0 y H1 **antes** de ver los resultados.
2. Elegir un **estadístico de prueba** que mida qué tan lejos están los datos de lo que predice H0. Por ejemplo, para dos medias, el estadístico *t* es la diferencia observada entre las medias **medida en errores estándar**: una *t* de 3 quiere decir que la diferencia es tres veces más grande que lo que el azar suele mover a una diferencia así.
3. Averiguar qué valores tomaría ese estadístico **si H0 fuera cierta** (con una fórmula, como la *t* de Student, o por simulación, como la permutación).
4. Calcular el valor *p*: la probabilidad, si H0 fuera cierta, de obtener un estadístico al menos tan alejado de cero como el observado. Si es menor que α (habitualmente 0.05), se rechaza H0.

**Ejemplo cotidiano:** un amigo dice que su moneda es normal (H0) y sale águila 10 veces seguidas. Si la moneda fuera normal, ver 10 águilas seguidas o 10 soles seguidos (igual de sospechoso) pasaría unas 2 veces en 1,000: ese es el valor *p*. No prueba que haga trampa, pero hace difícil creerle.

Si H0 es cierta y el estadístico de la prueba es continuo (como la *t* con datos de masa), el valor *p* puede salir con igual probabilidad en cualquier punto entre 0 y 1: hay 5% de probabilidad de que salga menor que 0.05 por puro azar. Eso es exactamente el error tipo I, y es lo que explota el *p-hacking*.

> 💡 **Rechazar no es lo mismo que aceptar.** Cuando *p* es grande no se «acepta» H0: se dice que los datos no muestran evidencia en su contra. Es como un juicio: la falta de pruebas no demuestra inocencia.

**Ejemplo: el valor *p* a la vista, por permutación.** Para ver qué significa «si H0 fuera cierta», se **revuelven al azar las etiquetas** macho y hembra de los 146 Adelie con sexo registrado y se calcula la diferencia de masa media en cada revuelta. Si el sexo no importara, esas diferencias mostrarían lo que el azar solo puede producir. Se hacen 10,000 revueltas y se compara con la diferencia observada.

* La siguiente celda revolverá 10,000 veces las etiquetas de sexo de los Adelie, dibujará las diferencias obtenidas junto a la diferencia observada (línea negra) y mostrará cuatro números: la diferencia observada, la desviación estándar de las revueltas, los límites entre los que quedó el 95% central de las revueltas y la mayor diferencia (en valor absoluto) que produjo el azar.

In [ ]:
adelie = pinguinos[pinguinos["species"] == "Adelie"].dropna(subset=["body_mass_g", "sex"])
masa_adelie = adelie["body_mass_g"].to_numpy()
es_macho = (adelie["sex"] == "Male").to_numpy()
diferencia_observada = masa_adelie[es_macho].mean() - masa_adelie[~es_macho].mean()

generador_permutacion = np.random.default_rng(401)
diferencias_revueltas = np.empty(10_000)
for i in range(10_000):
    revuelto = generador_permutacion.permutation(es_macho)
    diferencias_revueltas[i] = masa_adelie[revuelto].mean() - masa_adelie[~revuelto].mean()

ax = sns.histplot(diferencias_revueltas, bins=50)
ax.axvline(diferencia_observada, color="black")
ax.set(xlabel="diferencia de masa media (g) con las etiquetas revueltas",
       title="Lo que el azar produce si el sexo no importara (línea negra: diferencia observada)")
plt.show()
(round(float(diferencia_observada), 1), round(float(diferencias_revueltas.std(ddof=1)), 1),
 np.percentile(diferencias_revueltas, [2.5, 97.5]).round(1), round(float(np.abs(diferencias_revueltas).max()), 1))

**Interpretación:** la diferencia observada es de unos 675 g. Con las etiquetas revueltas, las diferencias se reparten alrededor de 0 con una desviación estándar de unos 76 g: el 95% central queda entre −152 y +149 g y la mayor que produjo el azar en 10,000 intentos fue de 314 g. La diferencia observada queda a casi nueve veces esa dispersión y ninguna revuelta la igualó, así que el valor *p* estimado es menor que 1 en 10,000 (un valor *p* nunca es exactamente 0, y con 10,000 revueltas no se puede precisar más): si el sexo no importara, no se vería una diferencia así. Eso, y nada más, es lo que significa un valor *p* pequeño.

**Ejemplo: comparaciones múltiples y *p-hacking*.** Se simularán 1,000 "estudios" en los que se comparan 20 variables entre dos grupos **idénticos** (H0 es cierta en todas). ¿En qué fracción de estudios aparece al menos un *p* < 0.05?

* La siguiente celda simulará los 1,000 estudios con 20 pruebas *t* cada uno, de forma vectorizada, con su propio generador de números aleatorios (semilla 401) para que el resultado se repita aunque se vuelva a ejecutar la celda.

In [ ]:
generador_estudios = np.random.default_rng(401)
grupo_a = generador_estudios.normal(size=(1000, 20, 30))
grupo_b = generador_estudios.normal(size=(1000, 20, 30))
valores_p = stats.ttest_ind(grupo_a, grupo_b, axis=2).pvalue
float((valores_p < 0.05).any(axis=1).mean())

Cerca de 64% de los estudios "encuentra algo" (1 − 0.95²⁰ ≈ 0.64, cuenta que supone pruebas independientes, como en esta simulación; con variables relacionadas entre sí saldría otra cifra) sin que exista ningún efecto. Probar muchas variables, subgrupos o versiones de un análisis y reportar solo lo significativo es *p-hacking*. **Decisiones:** definir las preguntas antes de mirar los resultados, reportar todas las pruebas hechas y, si son muchas, corregir con Holm o Bonferroni (`multipletests`).

* La siguiente celda aplicará la corrección de Holm a las 20 pruebas de cada estudio y calculará en qué fracción de estudios queda algún hallazgo.

In [ ]:
algun_hallazgo_holm = np.array([multipletests(p_estudio, alpha=0.05, method="holm")[0].any() for p_estudio in valores_p])
float(algun_hallazgo_holm.mean())

Con la corrección, la fracción de estudios con algún falso hallazgo vuelve a quedar alrededor de α = 5%. Ojo: la corrección no elimina los falsos hallazgos; solo mantiene en torno a α la probabilidad de cometer alguno en el conjunto de las 20 pruebas. El costo es menos potencia para cada prueba individual: por eso conviene hacer pocas pruebas, bien elegidas.

**Interpretación:** con Holm, solo ≈ 5.6% de los estudios (56 de 1,000) conserva algún falso hallazgo; la diferencia con 5% es ruido de simulación (con muchos más estudios sale ≈ 4.9%). En estos estudios, donde H0 es cierta en todas las pruebas, Bonferroni da exactamente la misma cifra (56 de 1,000). La idea de las correcciones es **exigir más** a cada prueba cuando se hacen muchas. Bonferroni, la más sencilla y estricta, divide el 0.05 entre el número de pruebas (con 20 pruebas, cada una necesita *p* < 0.0025). Holm hace lo mismo con el valor *p* más chico y va relajando la exigencia para los siguientes, así que pierde menos potencia cuando sí hay efectos verdaderos (aquí no los hay, y por eso ambas coinciden).

### 2.1 Potencia: ¿cuántos datos hacen falta?

La potencia depende del tamaño del efecto, de *n* y de α. `TTestIndPower` de *statsmodels* resuelve cualquiera de ellos dados los demás. El tamaño del efecto se expresa con la **d de Cohen** (diferencia de medias en unidades de desviación estándar).

**Concepto: d de Cohen.** Es la diferencia de medias medida en **desviaciones estándar** (combinando las de los dos grupos): una d de 0.5 quiere decir que los grupos están separados media desviación estándar. Como referencia aproximada, 0.2 es un efecto pequeño, 0.5 mediano y 0.8 grande.

**Ejemplo:** si dos grupos difieren 5 puntos en un cuestionario de práctica y los puntajes varían unos 10 puntos dentro de cada grupo, d ≈ 0.5.

Al no tener unidades, permite comparar efectos entre variables distintas y es el insumo de los cálculos de potencia.

* La siguiente celda calculará el *n* por grupo necesario para detectar d = 0.5 con 80% de potencia, y la potencia que se tendría con d = 0.3 y 30 por grupo.

In [ ]:
potencia = TTestIndPower()
(round(potencia.solve_power(effect_size=0.5, alpha=0.05, power=0.8), 1),
 round(float(potencia.power(effect_size=0.3, nobs1=30, alpha=0.05)), 3))

Se necesitan unos 64 casos por grupo para un efecto mediano; con 30 por grupo y un efecto pequeño (d = 0.3), solo se detectaría 1 de cada 5 veces. **Consecuencia:** en un estudio pequeño, "no significativo" casi no informa nada.

📖 **Para profundizar (opcional).** *Las fórmulas de la sección 2.*
* Estadístico *t* para dos medias: $t = \dfrac{\bar{x}_A - \bar{x}_B}{EE(\bar{x}_A - \bar{x}_B)}$, y valor *p* bilateral: $p = P(|T| \geq |t_{obs}| \mid H_0)$. Se rechaza H0 si $p < \alpha$.
* Con $m$ pruebas, Bonferroni compara cada valor *p* con $\alpha / m$; Holm ordena los valores *p* de menor a mayor y compara el *k*-ésimo con $\alpha / (m - k + 1)$.
* d de Cohen con la desviación estándar combinada $s_p$: $$d = \frac{\bar{x}_A - \bar{x}_B}{s_p}, \qquad s_p = \sqrt{\frac{(n_A - 1)s_A^2 + (n_B - 1)s_B^2}{n_A + n_B - 2}}$$

**💡 Ideas clave de la sección 2:**
* El valor *p* mide qué tan sorprendentes serían los datos **si H0 fuera cierta**; no es la probabilidad de H0.
* Con muchas pruebas aparecen falsos hallazgos: se planean pocas pruebas y se corrige (Holm).
* La potencia se planea antes: con *n* pequeño, un "no significativo" casi no informa.

### 🟢 Ejercicio 1 · ¿Qué dice un valor *p*?

**🎯 Objetivo:** distinguir la definición del valor *p* de sus lecturas erróneas más comunes. **Qué se espera aprender:** a no confundir *p* con la probabilidad de H0, con la magnitud del efecto ni con una prueba de igualdad.

Una prueba compara la masa de dos colonias y obtiene *p* = 0.03. Indica si cada afirmación es correcta: (a) "hay 3% de probabilidad de que H0 sea cierta"; (b) "si las colonias no difirieran, una diferencia así de grande o mayor ocurriría en torno a 3% de las veces"; (c) "la diferencia es grande"; (d) "con *p* = 0.30 se habría demostrado que las colonias son iguales".

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## ✅ Cierre.

**Práctica (en Classroom):** terminar el ejercicio 🟢 de este *notebook*.

**Siguiente *notebook*:** `20_pruebas_para_dos_grupos.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>